In [0]:
from pyspark.sql.functions import sum
from datetime import datetime
from pyspark.sql import Row
from uuid import uuid4

print("=" * 50)
print("GOLD START")
print("=" * 50)


def write_log(step, status, row_count, message):

    spark.createDataFrame([
        Row(run_id=str(uuid4()),
            run_timestamp=datetime.now(),
            step=step,
            status=status,
            row_count=row_count,
            message=message
        )
    ]).write \
      .mode("append") \
      .saveAsTable("training.monitoring.pipeline_logs")


try:

    # -------------------------
    # Lire Silver
    # -------------------------
    silver_df = spark.table(
        "training.silver.orders_clean"
    )

    # -------------------------
    # Transformation Gold
    # -------------------------
    gold_df = (
        silver_df
        .groupBy("country")
        .agg(
            sum("amount").alias("revenue")
        )
        .orderBy("revenue", ascending=False)
    )

    row_count = gold_df.count()

    gold_df.show()

    # -------------------------
    # Write Delta
    # -------------------------
    gold_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(
            "training.gold.revenue_by_country"
        )

    # -------------------------
    # Logs SUCCESS
    # -------------------------
    write_log(
        step="GOLD",
        status="SUCCESS",
        row_count=row_count,
        message="Aggregation revenue par country OK"
    )

    print(f"Gold rows: {row_count}")

except Exception as e:

    # -------------------------
    # Logs FAILED
    # -------------------------
    write_log(
        step="GOLD",
        status="FAILED",
        row_count=0,
        message=str(e)
    )

    raise


print("=" * 50)
print("GOLD END")
print("=" * 50)

